## Structural Data Splitting for End-to-End Evaluation

This notebook partitions nine structural datasets: the phase label
and eight structural targets (`a_1`, `b_1`, `c_1`, `Grain_1`,
`a_2`, `b_2`, `c_2`, and `Grain_2`).

### Workflow

1. Load the fixed Li-air outer compound split.
2. Inspect compound coverage across structural datasets.
3. Exclude Li-air outer-test compounds from each structural training pool.
4. Export matching outer-test records and a split summary.

Run after Li-air splitting and before structural model training.
Training pools retain all compounds outside the Li-air test set.
Matching uses exact formula strings after trimming whitespace.
This notebook does not generate cross-validation folds or predictions.

In [1]:
import os
import pandas as pd


STRUCTURE_FILES = {
    "label": "data/compound_6_Structural_label.csv",
    "Grain1": "data/compound_6_Structural_Grain1.csv",
    "a1": "data/compound_6_Structural_a1.csv",
    "b1": "data/compound_6_Structural_b1.csv",
    "c1": "data/compound_6_Structural_c1.csv",
    "Grain2": "data/compound_6_Structural_Grain2.csv",
    "a2": "data/compound_6_Structural_a2.csv",
    "b2": "data/compound_6_Structural_b2.csv",
    "c2": "data/compound_6_Structural_c2.csv"
}

GROUP_COL = "compound"

SAVE_DIR = "data/end_to_end_split/structure"
os.makedirs(SAVE_DIR, exist_ok=True)


# Load the fixed Li-air outer split
split_table = pd.read_csv(
    "data/end_to_end_split/outer_compound_split.csv"
)

split_table.columns = split_table.columns.str.strip()

split_table = split_table.rename(
    columns={"compounds": "compound"}
)

split_table[GROUP_COL] = (
    split_table[GROUP_COL]
    .astype(str)
    .str.strip()
)

air_train_compounds = set(
    split_table.loc[
        split_table["split"] == "train",
        GROUP_COL
    ]
)

air_test_compounds = set(
    split_table.loc[
        split_table["split"] == "test",
        GROUP_COL
    ]
)


# Load all structural datasets
structure_datasets = {}

for task_name, file_path in STRUCTURE_FILES.items():

    df = pd.read_csv(file_path)

    df = df.loc[
        :,
        ~df.columns.str.match(r"^Unnamed")
    ].copy()

    df.columns = df.columns.str.strip()

    if GROUP_COL not in df.columns:
        raise ValueError(
            f"The compound column is missing in {file_path}. "
            f"Available columns: {df.columns.tolist()}"
        )

    df[GROUP_COL] = (
        df[GROUP_COL]
        .astype(str)
        .str.strip()
    )

    structure_datasets[task_name] = df

    print(
        task_name,
        "rows:",
        len(df),
        "compounds:",
        df[GROUP_COL].nunique()
    )

label rows: 160 compounds: 156
Grain1 rows: 160 compounds: 156
a1 rows: 160 compounds: 156
b1 rows: 160 compounds: 156
c1 rows: 160 compounds: 156
Grain2 rows: 55 compounds: 55
a2 rows: 55 compounds: 55
b2 rows: 55 compounds: 55
c2 rows: 55 compounds: 55


In [2]:
reference_task = "label"

reference_compounds = set(
    structure_datasets[reference_task][GROUP_COL]
)

print("\nCompound-set comparison:")

for task_name, df in structure_datasets.items():

    task_compounds = set(df[GROUP_COL])

    missing_from_task = (
        reference_compounds
        - task_compounds
    )

    extra_in_task = (
        task_compounds
        - reference_compounds
    )

    print(
        task_name,
        "same set:",
        task_compounds == reference_compounds,
        "missing:",
        len(missing_from_task),
        "extra:",
        len(extra_in_task)
    )


Compound-set comparison:
label same set: True missing: 0 extra: 0
Grain1 same set: True missing: 0 extra: 0
a1 same set: True missing: 0 extra: 0
b1 same set: True missing: 0 extra: 0
c1 same set: True missing: 0 extra: 0
Grain2 same set: False missing: 101 extra: 0
a2 same set: False missing: 101 extra: 0
b2 same set: False missing: 101 extra: 0
c2 same set: False missing: 101 extra: 0


In [3]:
split_summary = []

for task_name, df in structure_datasets.items():

    structure_train_pool = df[
        ~df[GROUP_COL].isin(
            air_test_compounds
        )
    ].copy().reset_index(drop=True)

    structure_air_test = df[
        df[GROUP_COL].isin(
            air_test_compounds
        )
    ].copy().reset_index(drop=True)

    matched_test_compounds = set(
        structure_air_test[GROUP_COL]
    )

    missing_air_test_compounds = (
        air_test_compounds
        - matched_test_compounds
    )

    structure_train_pool.to_csv(
        os.path.join(
            SAVE_DIR,
            f"{task_name}_train_pool.csv"
        ),
        index=False
    )

    structure_air_test.to_csv(
        os.path.join(
            SAVE_DIR,
            f"{task_name}_air_test.csv"
        ),
        index=False
    )

    split_summary.append({
        "task": task_name,
        "train_rows": len(structure_train_pool),
        "train_compounds": (
            structure_train_pool[GROUP_COL].nunique()
        ),
        "air_test_rows": len(structure_air_test),
        "air_test_compounds": (
            structure_air_test[GROUP_COL].nunique()
        ),
        "missing_air_test_compounds": (
            len(missing_air_test_compounds)
        )
    })


split_summary = pd.DataFrame(split_summary)

split_summary.to_csv(
    os.path.join(
        SAVE_DIR,
        "structure_split_summary.csv"
    ),
    index=False
)

display(split_summary)

,task,train_rows,train_compounds,air_test_rows,air_test_compounds,missing_air_test_compounds
0,label,138,135,22,21,1
1,Grain1,138,135,22,21,1
2,a1,138,135,22,21,1
3,b1,138,135,22,21,1
4,c1,138,135,22,21,1
5,Grain2,47,47,8,8,14
6,a2,47,47,8,8,14
7,b2,47,47,8,8,14
8,c2,47,47,8,8,14
